# PhyloCartoPlot Workflow - Step-by-Step Walkthrough

This notebook walks through the complete pipeline for creating phylogeographic visualizations.

**Pipeline:**
1. Format GBIF coordinates data
2. Add caffeine content to GPS records
3. Build phylogenetic tree from sequences
4. Create tree + map visualization

## Setup

Define paths for input, scripts, and output folders.

In [ ]:
import os
import sys
from pathlib import Path

# Define paths relative to this notebook
workflow_dir = Path('..')
scripts_dir = workflow_dir / 'scripts'
input_dir = workflow_dir / 'input'
output_dir = workflow_dir / 'output'

# Add scripts directory to Python path so we can import modules
sys.path.insert(0, str(scripts_dir))

print(f"Workflow directory: {workflow_dir.absolute()}")
print(f"Scripts: {scripts_dir.absolute()}")
print(f"Input: {input_dir.absolute()}")
print(f"Output: {output_dir.absolute()}")
print(f"\n✓ Paths configured")

## Step 1: Format GBIF Data

Take GBIF coordinates and map them to phylogenetic node names.

In [ ]:
from format_gbif_data import format_gbif

print("Step 1: Formatting GBIF data...\n")

gbif_file = input_dir / 'coords_w_caff.csv'
nodes_file = input_dir / 'node_names.csv'

print(f"Input files:")
print(f"  - GBIF: {gbif_file.name}")
print(f"  - Nodes: {nodes_file.name}")

# Change to scripts directory to match expected paths in script
os.chdir(scripts_dir)

# Call format_gbif
format_gbif(str(gbif_file), str(nodes_file))

# Change back to notebook directory
os.chdir(workflow_dir / 'notebooks')

print(f"\n✓ Step 1 complete")

## Step 2: Add Caffeine Content

Merge formatted GPS data with caffeine content.

In [ ]:
from add_caffeine_coords import add_caffeine

print("Step 2: Adding caffeine content...\n")

formatted_file = input_dir / 'coords_w_caff_formatted.csv'
caffeine_file = input_dir / 'no_caffeine_nodes_w_specimen.csv'

print(f"Input files:")
print(f"  - Formatted GPS: {formatted_file.name}")
print(f"  - Caffeine data: {caffeine_file.name}")

# Change to scripts directory
os.chdir(scripts_dir)

# Call add_caffeine
add_caffeine(str(formatted_file), str(caffeine_file))

# Change back
os.chdir(workflow_dir / 'notebooks')

print(f"\n✓ Step 2 complete")

## Step 3: Build Phylogenetic Tree

Build tree from aligned sequences.

In [ ]:
from build_phylogenetic_tree import build_tree

print("Step 3: Building phylogenetic tree...\n")

fasta_file = input_dir / 'aligned_caffeine.fasta'

print(f"Input file:")
print(f"  - FASTA: {fasta_file.name}")

# Change to scripts directory
os.chdir(scripts_dir)

# Call build_tree
build_tree(str(fasta_file))

# Change back
os.chdir(workflow_dir / 'notebooks')

print(f"\n✓ Step 3 complete")

## Step 4: Create Visualization

Create the final tree + map visualization.

In [ ]:
print("Step 4: Creating phylogeographic visualization...\n")

# Files generated in previous steps
tree_file = input_dir / 'aligned_caffeine_tree.nwk'
gps_file = input_dir / 'coords_w_caff_w_caffeine.csv'
offsets_file = input_dir / 'offsets_caff.csv'

print(f"Input files:")
print(f"  - Tree: {tree_file.name}")
print(f"  - GPS + Caffeine: {gps_file.name}")
print(f"  - Offsets: {offsets_file.name}")

# Check if files exist
if not tree_file.exists():
    print(f"\n⚠ Warning: {tree_file.name} not found")
    print(f"Make sure Step 3 completed successfully")
else:
    print(f"\n✓ All input files found")
    print(f"\nTo run visualization, execute from command line:")
    print(f"cd scripts")
    print(f"python tree_to_map_caffeine_content.py --nwk {tree_file} --gps {gps_file} --offset {offsets_file}")

## Summary

The pipeline has completed the first three steps:

In [ ]:
import pandas as pd

print("="*70)
print("PIPELINE SUMMARY")
print("="*70)

# Check what files were created
print("\nGenerated files in input/ directory:")
for f in sorted(input_dir.glob('*')):
    if f.is_file():
        size = f.stat().st_size
        if size > 1024*1024:
            size_str = f"{size/(1024*1024):.1f} MB"
        elif size > 1024:
            size_str = f"{size/1024:.1f} KB"
        else:
            size_str = f"{size} B"
        print(f"  {f.name:40} {size_str:>12}")

# Show summary of key files
print(f"\n" + "="*70)
print("KEY OUTPUTS:")
print("="*70)

formatted_file = input_dir / 'coords_w_caff_formatted.csv'
if formatted_file.exists():
    df = pd.read_csv(formatted_file)
    print(f"\n✓ Formatted GPS data ({formatted_file.name}):")
    print(f"  - Records: {len(df)}")
    print(f"  - Columns: {', '.join(df.columns)}")
    print(df.head(3))

caffeine_file = input_dir / 'coords_w_caff_w_caffeine.csv'
if caffeine_file.exists():
    df = pd.read_csv(caffeine_file)
    print(f"\n✓ GPS + Caffeine data ({caffeine_file.name}):")
    print(f"  - Records: {len(df)}")
    print(f"  - Columns: {', '.join(df.columns)}")
    print(f"  - Caffeine range: {df['caffeine_percent'].min():.3f} - {df['caffeine_percent'].max():.3f}%")
    print(df.head(3))

tree_file = input_dir / 'aligned_caffeine_tree.nwk'
if tree_file.exists():
    print(f"\n✓ Phylogenetic tree ({tree_file.name}):")
    with open(tree_file) as f:
        tree_content = f.read()
    print(f"  - Tree generated successfully")
    print(f"  - Size: {len(tree_content)} characters")

print(f"\n" + "="*70)
print("NEXT STEP:")
print("="*70)
print(f"\nRun tree_to_map_caffeine_content.py to create visualization")
print(f"\nCommand:")
print(f"cd scripts")
print(f"python tree_to_map_caffeine_content.py \\")
print(f"    --nwk ../input/aligned_caffeine_tree.nwk \\")
print(f"    --gps ../input/coords_w_caff_w_caffeine.csv \\")
print(f"    --offset ../input/offsets_caff.csv")

## Output Files

After running all steps, the following files will be generated:

**In `input/` folder:**
- `coords_w_caff_formatted.csv` - Formatted GPS coordinates
- `coords_w_caff_w_caffeine.csv` - GPS coordinates with caffeine content
- `aligned_caffeine_tree.nwk` - Phylogenetic tree in Newick format

**In `output/` folder (after step 4):**
- `tree2map.svg` - Vector visualization
- `tree2map.png` - Raster visualization